# Cluster and Visualize Unlabeled Acoustic Embeddings (HDBSCAN)

This notebook:
1. Loads the 1536-dim Perch embeddings
2. Reduces to **3D** with **UMAP** (full) and **t-SNE** (subset) for visualization
3. Clusters with **HDBSCAN** (automatic number of clusters + noise handling)
4. Samples audio files per cluster (nearest + farthest + random)
5. Saves audios to Review folder for manual QA

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from os import environ

import umap
from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from tqdm import tqdm

import hdbscan

import shutil

## 1. Load Embeddings and Manifest

In [4]:
env_dir = environ.get("POSIDONIA_DATASET_DIR")
DATASET_DIR = Path(env_dir) if env_dir else Path("D:/Posidonia Soundscapes/Fondeo 1_Formentera Ille Espardell/Embeddings_2/dataset")

EMBEDDINGS_PATH = DATASET_DIR / "npy_files" / "unlabeled_embeddings.npy"
MANIFEST_PATH = DATASET_DIR / "unlabeled_manifest.csv"

if not EMBEDDINGS_PATH.exists() or not MANIFEST_PATH.exists():
    raise FileNotFoundError(
        f"Missing files:\n"
        f"  embeddings: {EMBEDDINGS_PATH}\n"
        f"  manifest:   {MANIFEST_PATH}\n"
        f"Set POSIDONIA_DATASET_DIR to the correct dataset folder."
    )

embeddings = np.load(str(EMBEDDINGS_PATH))
manifest_df = pd.read_csv(str(MANIFEST_PATH))

print(f"Loaded {embeddings.shape[0]} embeddings of dimension {embeddings.shape[1]}")
print(f"Manifest has {len(manifest_df)} rows")
manifest_df.head()

Loaded 392400 embeddings of dimension 1536
Manifest has 392400 rows


,original_audio,embedding_path,segment_path,audio_path,file_name,embedding_dim
0,/mnt/d/Posidonia Soundscapes/Fondeo 1_Formente...,/mnt/d/Posidonia Soundscapes/Fondeo 1_Formente...,NaN,/mnt/d/Posidonia Soundscapes/Fondeo 1_Formente...,channelA_2025-05-16_14-00-03.wav,1536
1,/mnt/d/Posidonia Soundscapes/Fondeo 1_Formente...,/mnt/d/Posidonia Soundscapes/Fondeo 1_Formente...,NaN,/mnt/d/Posidonia Soundscapes/Fondeo 1_Formente...,channelA_2025-05-16_14-00-08.wav,1536
2,/mnt/d/Posidonia Soundscapes/Fondeo 1_Formente...,/mnt/d/Posidonia Soundscapes/Fondeo 1_Formente...,NaN,/mnt/d/Posidonia Soundscapes/Fondeo 1_Formente...,channelA_2025-05-16_14-00-13.wav,1536
3,/mnt/d/Posidonia Soundscapes/Fondeo 1_Formente...,/mnt/d/Posidonia Soundscapes/Fondeo 1_Formente...,NaN,/mnt/d/Posidonia Soundscapes/Fondeo 1_Formente...,channelA_2025-05-16_14-00-18.wav,1536
4,/mnt/d/Posidonia Soundscapes/Fondeo 1_Formente...,/mnt/d/Posidonia Soundscapes/Fondeo 1_Formente...,NaN,/mnt/d/Posidonia Soundscapes/Fondeo 1_Formente...,channelA_2025-05-16_14-00-23.wav,1536


## 2. Dimensionality Reduction: UMAP and t-SNE to 3D

In [5]:
output_dir = EMBEDDINGS_PATH.parent
umap_tsne_output_dir = output_dir / "umap_and_tsne"
umap_tsne_output_dir.mkdir(parents=True, exist_ok=True)

umap_file = umap_tsne_output_dir / "umap_embeddings.npy"
tsne_file = umap_tsne_output_dir / "tsne_subset_embeddings.npy"
tsne_idx_file = umap_tsne_output_dir / "tsne_subset_indices.npy"

if umap_file.exists() and tsne_file.exists() and tsne_idx_file.exists():
    print("Loading pre-computed UMAP and t-SNE embeddings...")
    umap_results = np.load(str(umap_file))
    tsne_results = np.load(str(tsne_file))
    tsne_idx = np.load(str(tsne_idx_file))
else:
    print("Pre-reducing to 256D with PCA to avoid memory crashes...")
    embeddings_fp32 = embeddings.astype(np.float32, copy=False)
    embeddings_pca_256 = PCA(n_components=256, random_state=42).fit_transform(embeddings_fp32)

    print("Running UMAP on 256D embeddings...")
    umap_model = umap.UMAP(
        n_components=3,
        random_state=42,
        n_neighbors=15,
        min_dist=0.1,
        metric="euclidean",
        low_memory=True
    )
    umap_results = umap_model.fit_transform(embeddings_pca_256)

    print("Running t-SNE on subset for visualization...")
    TSNE_VIS_LIMIT = 50000
    rng = np.random.default_rng(42)
    tsne_idx = rng.choice(embeddings.shape[0], size=min(TSNE_VIS_LIMIT, embeddings.shape[0]), replace=False)

    tsne = TSNE(n_components=3, random_state=42, perplexity=30)
    tsne_results = tsne.fit_transform(embeddings_pca_256[tsne_idx])

    np.save(str(umap_file), umap_results)
    np.save(str(tsne_file), tsne_results)
    np.save(str(tsne_idx_file), tsne_idx)

manifest_df["umap_x"] = umap_results[:, 0]
manifest_df["umap_y"] = umap_results[:, 1]
manifest_df["umap_z"] = umap_results[:, 2]

manifest_df["tsne_x"] = np.nan
manifest_df["tsne_y"] = np.nan
manifest_df["tsne_z"] = np.nan
manifest_df.loc[tsne_idx, "tsne_x"] = tsne_results[:, 0]
manifest_df.loc[tsne_idx, "tsne_y"] = tsne_results[:, 1]
manifest_df.loc[tsne_idx, "tsne_z"] = tsne_results[:, 2]

print("UMAP and t-SNE embeddings added to manifest.")

Loading pre-computed UMAP and t-SNE embeddings...
UMAP and t-SNE embeddings added to manifest.


## 3. HDBSCAN Clustering on UMAP

In [6]:
HDBSCAN_MIN_CLUSTER_SIZE = 80
HDBSCAN_MIN_SAMPLES = 10

umap_labels_file = umap_tsne_output_dir / f"umap_hdbscan_labels_mcs{HDBSCAN_MIN_CLUSTER_SIZE}_ms{HDBSCAN_MIN_SAMPLES}.npy"

if umap_labels_file.exists():
    print("Loading pre-computed UMAP HDBSCAN clustering...")
    umap_labels = np.load(str(umap_labels_file))
else:
    print("Running HDBSCAN clustering on UMAP embeddings...")
    clusterer_umap = hdbscan.HDBSCAN(
        min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
        min_samples=HDBSCAN_MIN_SAMPLES,
        metric="euclidean",
        cluster_selection_method="eom"
    )
    umap_labels = clusterer_umap.fit_predict(umap_results)
    np.save(str(umap_labels_file), umap_labels)

manifest_df["umap_cluster"] = umap_labels

unique_umap = np.unique(umap_labels)
n_umap_clusters = np.sum(unique_umap >= 0)
n_umap_noise = np.sum(umap_labels == -1)
print(f"UMAP HDBSCAN clusters found: {n_umap_clusters}")
print(f"UMAP noise points: {n_umap_noise}")
print(pd.Series(umap_labels).value_counts().sort_index())

Running HDBSCAN clustering on UMAP embeddings...
UMAP HDBSCAN clusters found: 5
UMAP noise points: 38
-1        38
 0       116
 1       111
 2    391888
 3       155
 4        92
Name: count, dtype: int64


## 4. HDBSCAN Clustering on t-SNE

In [7]:
tsne_labels_file = umap_tsne_output_dir / f"tsne_subset_hdbscan_labels_mcs{HDBSCAN_MIN_CLUSTER_SIZE}_ms{HDBSCAN_MIN_SAMPLES}.npy"

if tsne_labels_file.exists():
    print("Loading pre-computed t-SNE HDBSCAN clustering...")
    tsne_labels = np.load(str(tsne_labels_file))
else:
    print("Running HDBSCAN clustering on t-SNE subset embeddings...")
    clusterer_tsne = hdbscan.HDBSCAN(
        min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
        min_samples=HDBSCAN_MIN_SAMPLES,
        metric="euclidean",
        cluster_selection_method="eom"
    )
    tsne_labels = clusterer_tsne.fit_predict(tsne_results)
    np.save(str(tsne_labels_file), tsne_labels)

manifest_df["tsne_cluster"] = np.nan
manifest_df.loc[tsne_idx, "tsne_cluster"] = tsne_labels

unique_tsne = np.unique(tsne_labels)
n_tsne_clusters = np.sum(unique_tsne >= 0)
n_tsne_noise = np.sum(tsne_labels == -1)
print(f"t-SNE HDBSCAN clusters found: {n_tsne_clusters}")
print(f"t-SNE noise points: {n_tsne_noise}")
print(pd.Series(tsne_labels).value_counts().sort_index())

Running HDBSCAN clustering on t-SNE subset embeddings...
t-SNE HDBSCAN clusters found: 3
t-SNE noise points: 22153
-1    22153
 0       91
 1      103
 2    27653
Name: count, dtype: int64


## 5. Sample Strategy per Cluster

For each non-noise cluster, sample nearest + farthest + random points.

In [8]:
def sample_cluster(cluster_indices, cluster_embeddings, n_nearest=20, n_farthest=20, n_random=60):
    if len(cluster_indices) == 0:
        return np.array([], dtype=int)

    centroid = np.mean(cluster_embeddings, axis=0)
    dists = np.linalg.norm(cluster_embeddings - centroid, axis=1)
    order = np.argsort(dists)

    n_nearest_eff = min(n_nearest, len(order))
    nearest_idx = cluster_indices[order[:n_nearest_eff]]

    remaining_after_nearest = np.setdiff1d(cluster_indices, nearest_idx)
    n_farthest_eff = min(n_farthest, len(remaining_after_nearest))
    farthest_idx = cluster_indices[order[-n_farthest_eff:]] if n_farthest_eff > 0 else np.array([], dtype=int)

    used_idx = np.unique(np.concatenate([nearest_idx, farthest_idx]))
    remaining_idx = np.setdiff1d(cluster_indices, used_idx)

    n_random_eff = min(n_random, len(remaining_idx))
    if n_random_eff > 0:
        rng = np.random.default_rng(42)
        random_idx = rng.choice(remaining_idx, size=n_random_eff, replace=False)
    else:
        random_idx = np.array([], dtype=int)

    sampled_idx = np.unique(np.concatenate([nearest_idx, farthest_idx, random_idx]))
    return sampled_idx

umap_csv = umap_tsne_output_dir / f"subsample_umap_hdbscan_mcs{HDBSCAN_MIN_CLUSTER_SIZE}_ms{HDBSCAN_MIN_SAMPLES}.csv"
tsne_csv = umap_tsne_output_dir / f"subsample_tsne_hdbscan_mcs{HDBSCAN_MIN_CLUSTER_SIZE}_ms{HDBSCAN_MIN_SAMPLES}.csv"

if umap_csv.exists() and tsne_csv.exists():
    print("Loading pre-computed sample indices...")
    umap_subsample_df = pd.read_csv(str(umap_csv))
    tsne_subsample_df = pd.read_csv(str(tsne_csv))
    umap_subsample_indices = umap_subsample_df["reduced_embeddings_idx"].values
    tsne_subsample_indices = tsne_subsample_df["reduced_embeddings_idx"].values
else:
    print("Sampling from UMAP clusters...")
    umap_subsample_indices = []
    umap_cluster_ids = [c for c in np.unique(umap_labels) if c >= 0]
    for cluster_id in tqdm(umap_cluster_ids):
        cluster_indices = np.where(umap_labels == cluster_id)[0]
        cluster_embeddings = umap_results[cluster_indices]
        sampled = sample_cluster(cluster_indices, cluster_embeddings)
        umap_subsample_indices.extend(sampled.tolist())
    umap_subsample_indices = np.unique(umap_subsample_indices)

    print("Sampling from t-SNE clusters...")
    tsne_subsample_indices = []
    tsne_cluster_ids = [c for c in np.unique(tsne_labels) if c >= 0]
    for cluster_id in tqdm(tsne_cluster_ids):
        cluster_indices_subset = np.where(tsne_labels == cluster_id)[0]
        cluster_embeddings_subset = tsne_results[cluster_indices_subset]
        sampled_subset = sample_cluster(cluster_indices_subset, cluster_embeddings_subset)
        sampled_orig = tsne_idx[sampled_subset]
        tsne_subsample_indices.extend(sampled_orig.tolist())
    tsne_subsample_indices = np.unique(tsne_subsample_indices)

print(f"Selected {len(umap_subsample_indices)} UMAP samples")
print(f"Selected {len(tsne_subsample_indices)} t-SNE samples")

Sampling from UMAP clusters...


100%|██████████| 5/5 [00:00<00:00, 119.12it/s]


Sampling from t-SNE clusters...


100%|██████████| 3/3 [00:00<00:00, 802.64it/s]

Selected 492 UMAP samples
Selected 291 t-SNE samples


In [9]:
umap_subsample_df = pd.DataFrame({
    "audio_path": manifest_df.iloc[umap_subsample_indices]["audio_path"].values,
    "embedding_path": manifest_df.iloc[umap_subsample_indices]["embedding_path"].values,
    "reduced_embedding_filepath": str(umap_file),
    "reduced_embeddings_idx": umap_subsample_indices,
    "method": "umap_hdbscan",
    "cluster": umap_labels[umap_subsample_indices]
})

tsne_subsample_df = pd.DataFrame({
    "audio_path": manifest_df.iloc[tsne_subsample_indices]["audio_path"].values,
    "embedding_path": manifest_df.iloc[tsne_subsample_indices]["embedding_path"].values,
    "reduced_embedding_filepath": str(tsne_file),
    "reduced_embeddings_idx": tsne_subsample_indices,
    "method": "tsne_hdbscan",
    "cluster": manifest_df.iloc[tsne_subsample_indices]["tsne_cluster"].astype(int).values
})

print(f"UMAP subsample: {len(umap_subsample_df)} samples")
print(f"t-SNE subsample: {len(tsne_subsample_df)} samples")

UMAP subsample: 492 samples
t-SNE subsample: 291 samples


## 6. Save Results

In [10]:
combined_csv = umap_tsne_output_dir / f"subsample_combined_hdbscan_mcs{HDBSCAN_MIN_CLUSTER_SIZE}_ms{HDBSCAN_MIN_SAMPLES}.csv"
combined_subsample_df = pd.concat([umap_subsample_df, tsne_subsample_df], ignore_index=True)

umap_subsample_df.to_csv(umap_csv, index=False)
tsne_subsample_df.to_csv(tsne_csv, index=False)
combined_subsample_df.to_csv(combined_csv, index=False)

np.save(umap_labels_file, umap_labels)
np.save(tsne_labels_file, tsne_labels)
np.save(tsne_idx_file, tsne_idx)

print(f"Saved UMAP subsample: {umap_csv}")
print(f"Saved t-SNE subsample: {tsne_csv}")
print(f"Saved combined subsample: {combined_csv}")

Saved UMAP subsample: D:\Posidonia Soundscapes\Fondeo 1_Formentera Ille Espardell\Embeddings_2\dataset\npy_files\umap_and_tsne\subsample_umap_hdbscan_mcs80_ms10.csv
Saved t-SNE subsample: D:\Posidonia Soundscapes\Fondeo 1_Formentera Ille Espardell\Embeddings_2\dataset\npy_files\umap_and_tsne\subsample_tsne_hdbscan_mcs80_ms10.csv
Saved combined subsample: D:\Posidonia Soundscapes\Fondeo 1_Formentera Ille Espardell\Embeddings_2\dataset\npy_files\umap_and_tsne\subsample_combined_hdbscan_mcs80_ms10.csv


## 7. Copy Audio Samples to Review Folder

In [11]:
def convert_wsl_to_windows_path(wsl_path):
    if isinstance(wsl_path, float):
        return wsl_path
    wsl_path = str(wsl_path).replace('\\', '/')
    if wsl_path.startswith('/mnt/') and len(wsl_path) > 7:
        drive = wsl_path[5].upper()
        rest = wsl_path[7:]
        return f"{drive}:\\{rest}".replace('/', '\\')
    return wsl_path

REVIEW_DIR = Path('D:/Posidonia Soundscapes/Fondeo 1_Formentera Ille Espardell/Embeddings_2/diagnostics/Review_HDBSCAN')
UMAP_DEST = REVIEW_DIR / 'UMAP'
TSNE_DEST = REVIEW_DIR / 't-SNE'

UMAP_ALL = UMAP_DEST / 'All'
TSNE_ALL = TSNE_DEST / 'All'
UMAP_CLUSTERS = UMAP_DEST / 'Clusters'
TSNE_CLUSTERS = TSNE_DEST / 'Clusters'

for folder in [UMAP_ALL, TSNE_ALL, UMAP_CLUSTERS, TSNE_CLUSTERS]:
    folder.mkdir(parents=True, exist_ok=True)

def cluster_folder_name(cluster_label):
    cluster_label = int(cluster_label)
    return 'Noise' if cluster_label == -1 else str(cluster_label + 1)

for cluster_label in sorted(umap_subsample_df['cluster'].dropna().astype(int).unique()):
    (UMAP_CLUSTERS / cluster_folder_name(cluster_label)).mkdir(exist_ok=True)

for cluster_label in sorted(tsne_subsample_df['cluster'].dropna().astype(int).unique()):
    (TSNE_CLUSTERS / cluster_folder_name(cluster_label)).mkdir(exist_ok=True)

def copy_audio_files(df, all_dest, cluster_dest_root):
    copied, failed = 0, 0
    for _, row in df.iterrows():
        src_path = Path(convert_wsl_to_windows_path(row['audio_path']))
        if not src_path.exists():
            failed += 1
            continue

        cluster_dir = cluster_dest_root / cluster_folder_name(row['cluster'])
        dst_all = all_dest / src_path.name
        dst_cluster = cluster_dir / src_path.name

        try:
            shutil.copy2(src_path, dst_all)
            shutil.copy2(src_path, dst_cluster)
            copied += 1
        except Exception:
            failed += 1
    return copied, failed

umap_copied, umap_failed = copy_audio_files(umap_subsample_df, UMAP_ALL, UMAP_CLUSTERS)
tsne_copied, tsne_failed = copy_audio_files(tsne_subsample_df, TSNE_ALL, TSNE_CLUSTERS)

print(f"UMAP copied: {umap_copied}, failed: {umap_failed}")
print(f"t-SNE copied: {tsne_copied}, failed: {tsne_failed}")
print(f"Review folder: {REVIEW_DIR}")

UMAP copied: 492, failed: 0
t-SNE copied: 291, failed: 0
Review folder: D:\Posidonia Soundscapes\Fondeo 1_Formentera Ille Espardell\Embeddings_2\diagnostics\Review_HDBSCAN
